# 138 · Vision Q&A Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/138-vision-qa-agent/vision_qa_agent_workbook.ipynb)

**Source:** `examples/138-vision-qa-agent/`

**What you'll learn:**
- How vision models tile images into 512 px crops and charge image tokens
- The difference between public URL inputs and inline base64 encoding
- How to build the OpenAI multimodal `content` array with mixed image + text blocks
- How to route different image types to specialised prompt templates
- How a LangGraph single-node `VisionState` graph wraps the whole pipeline

**Requirements:**
- Part A (cells 1–12): no API key — pure Python, runs anywhere
- Part B (cells 13–18): requires `OPENAI_API_KEY` in `.env`; three live vision requests

**Models used:** `gpt-5.4-nano` (vision-capable)

In [ ]:
%pip install -q openai httpx python-dotenv langgraph

---
## Part A — CPU-Safe Concept Demonstrations

No API key needed. Every cell in Part A runs on plain Python + `httpx`.
You will build and inspect every component of the vision pipeline before spending a single token.

### Concept 1: What are vision tokens?

Vision-capable models do not see a raw image file. They first **tile** the image into crops, then encode each tile as image tokens. The `detail` parameter on the `image_url` block controls how many tiles are used:

| Detail level | Image size | Tiles | Approx tokens |
|---|---|---|---|
| `low` | any | 1 | ~85 |
| `auto` (small) | ≤ 512 × 512 | 1 | ~255 |
| `auto` (medium) | ~ 1024 × 768 | 6 | ~1 105 |
| `high` | 2048 × 2048 | 13 + | ~2 295 + |

**Takeaway:** Large images at `high` detail can cost more in tokens than the text question and answer combined. For most Q&A tasks, `auto` or `low` is the right default.

In [ ]:
# Concept 1b — Token formula in code
# Reproduce the tile calculation from the OpenAI docs.

import math


def estimate_vision_tokens(width: int, height: int, detail: str = "auto") -> int:
    """Estimate the image-token cost before making an API call."""
    if detail == "low":
        return 85
    # Step 1: scale to fit 2048 x 2048
    if max(width, height) > 2048:
        scale = 2048 / max(width, height)
        width, height = int(width * scale), int(height * scale)
    # Step 2: scale so shortest side <= 768
    if min(width, height) > 768:
        scale = 768 / min(width, height)
        width, height = int(width * scale), int(height * scale)
    # Step 3: count 512 x 512 tiles
    tiles = math.ceil(width / 512) * math.ceil(height / 512)
    return tiles * 170 + 85


test_cases = [
    (240,  240,  "low"),
    (240,  240,  "auto"),
    (512,  512,  "high"),
    (1024, 768,  "high"),
    (2048, 2048, "high"),
]

print(f"{'Width':>6}  {'Height':>6}  {'Detail':<6}  {'Tokens':>8}")
print("-" * 36)
for w, h, d in test_cases:
    t = estimate_vision_tokens(w, h, d)
    print(f"{w:>6}  {h:>6}  {d:<6}  {t:>8}")

### Concept 2: base64 vs URL inputs

The OpenAI vision API accepts images in two forms:

| Method | Pros | Cons | Best for |
|---|---|---|---|
| Public URL | Zero overhead, fast, no body bloat | Image must be publicly reachable | Demos, public assets |
| base64 inline | Works offline / behind auth, private images | +33 % size, request body grows | Private images, local files |
| File path | Simple local dev | Only works on the local machine | Local dev only |

Both URL and base64 use the same `image_url` block — you just change the `url` field value:

```python
# Public URL form
{"type": "image_url", "image_url": {"url": "https://example.com/photo.jpg"}}

# Base64 inline form
{"type": "image_url", "image_url": {"url": "data:image/jpeg;base64,/9j/4AAQ..."}}
```

The **data URI** format is: `data:<mime-type>;base64,<base64-encoded-bytes>`

### Concept 3: The content array structure

A standard OpenAI chat message uses a string for `content`. Vision messages use a **list of typed blocks** instead:

```json
{
  "role": "user",
  "content": [
    {"type": "image_url", "image_url": {"url": "data:image/png;base64,..."}},
    {"type": "text", "text": "What is in this image?"}
  ]
}
```

ASCII diagram of a single-image vision message:

```
messages[0]
├── role: "user"
└── content: list
    ├── [0] {"type": "image_url", "image_url": {"url": "data:image/png;base64,..."}}
    └── [1] {"type": "text", "text": "What is in this image?"}
```

Rules:
- Put the **image block first**, text question second — the model reads image before text.
- You can include **multiple** `image_url` blocks in the same `content` list.
- The optional `"detail"` key on `image_url` controls the tile budget (see Concept 1).

In [ ]:
# MIME detection — the mime_map from src/tools.py
# No API call needed.

from pathlib import Path

MIME_MAP = {
    "jpg":  "image/jpeg",
    "jpeg": "image/jpeg",
    "png":  "image/png",
    "webp": "image/webp",
    "gif":  "image/gif",
}

test_filenames = [
    "photo.jpg",
    "diagram.PNG",   # uppercase extension
    "screenshot.webp",
    "animation.gif",
    "unknown.bmp",   # not in map — falls back to jpeg
]

print(f"{'Filename':<25} {'Detected MIME':<20}")
print("-" * 46)
for fname in test_filenames:
    ext = Path(fname).suffix.lstrip(".").lower()
    mime = MIME_MAP.get(ext, "image/jpeg")   # fallback matches tools.py
    print(f"{fname:<25} {mime:<20}")

In [ ]:
# Fetch a public image and encode it to base64
# No API key needed — just httpx.

import base64
import httpx

IMAGE_URL = (
    "https://raw.githubusercontent.com/github/explore/main/topics/"
    "python/python.png"
)

resp = httpx.get(IMAGE_URL, timeout=10)
resp.raise_for_status()

mime = resp.headers.get("content-type", "image/png").split(";")[0]
raw_bytes = resp.content
b64_string = base64.b64encode(raw_bytes).decode()

overhead_pct = (len(b64_string) / len(raw_bytes) - 1) * 100
data_uri = f"data:{mime};base64,{b64_string}"

print(f"Raw bytes        : {len(raw_bytes):>8,}")
print(f"Base64 chars     : {len(b64_string):>8,}")
print(f"Size overhead    : {overhead_pct:>7.1f} %")
print(f"First 80 b64 chars: {b64_string[:80]}")
print(f"\nData URI prefix  : {data_uri[:60]}...")

In [ ]:
# Build a full vision payload WITHOUT making an API call.
# This mirrors exactly what src/tools.py vision_content() does.

import json

QUESTION = "What colors appear in this image?"

content = [
    {"type": "image_url", "image_url": {"url": data_uri}},
    {"type": "text", "text": QUESTION},
]

message = {"role": "user", "content": content}

# Print with b64 truncated for readability
display_msg = {
    "role": message["role"],
    "content": [
        {
            "type": content[0]["type"],
            "image_url": {"url": data_uri[:55] + "... [truncated]"},
        },
        content[1],
    ],
}

print("Vision message structure:")
print(json.dumps(display_msg, indent=2))

print(f"\nTotal content blocks  : {len(content)}")
print(f"Image block url length: {len(data_uri):,} chars (full b64 URI)")

### Concept 4: Image routing by type

A real vision agent often knows — or can infer — what kind of image it is processing.  
Different image types benefit from different prompt templates that direct the model's attention:

| Image type | Prompt template focus |
|---|---|
| chart / graph | "Describe the trend, axes labels, and key values shown." |
| photo | "Describe the scene, subjects, and mood." |
| diagram / flowchart | "Explain the steps or relationships shown." |
| screenshot / UI | "Identify UI elements, buttons, and any visible text." |
| document / text | "Extract and transcribe all visible text exactly." |

This routing layer is zero-cost — it runs entirely in Python before any API call.

In [ ]:
# Simulate vision routing — no API key needed.

PROMPT_TEMPLATES = {
    "chart": "Describe the trend, axes labels, and key values shown in this chart.",
    "photo": "Describe the scene, main subjects, and overall mood.",
    "diagram": "Explain the steps or relationships shown in this diagram.",
    "screenshot": "Identify all UI elements, buttons, and any visible text.",
    "document": "Extract and transcribe all visible text exactly as it appears.",
}


def get_prompt(image_type: str, fallback: str = "Describe what you see.") -> str:
    return PROMPT_TEMPLATES.get(image_type.lower(), fallback)


print(f"{'Image type':<15} {'Prompt sent to model':<55}")
print("-" * 70)
for itype in ["chart", "photo", "diagram", "screenshot", "document", "unknown"]:
    prompt = get_prompt(itype)
    print(f"{itype:<15} {prompt[:55]}")

In [ ]:
# Build a multi-image payload — two images, one text block.
# No API call; just shows how to extend the content list.

SECOND_IMAGE_URL = "https://raw.githubusercontent.com/github/explore/main/topics/rust/rust.png"

# Fetch second image
resp2 = httpx.get(SECOND_IMAGE_URL, timeout=10)
resp2.raise_for_status()
mime2 = resp2.headers.get("content-type", "image/jpeg").split(";")[0]
b64_2 = base64.b64encode(resp2.content).decode()
data_uri2 = f"data:{mime2};base64,{b64_2}"

# Build content with two images
multi_content = [
    {"type": "image_url", "image_url": {"url": data_uri}},
    {"type": "image_url", "image_url": {"url": data_uri2}},
    {"type": "text", "text": "Compare these two images. What is different about them?"},
]

print(f"Content blocks  : {len(multi_content)}")
print(f"Block types     : {[b['type'] for b in multi_content]}")
print(f"Image 1 MIME    : {mime}   ({len(b64_string):,} chars)")
print(f"Image 2 MIME    : {mime2}  ({len(b64_2):,} chars)")
print(f"Text question   : '{multi_content[2]['text']}'")
print("\nKey insight: both images travel in ONE API call — more token-efficient")
print("than two separate calls because prompt + system overhead is paid once.")

---
### Exercise 1: Send Two Images in One Request

The current `vision_content()` in `src/tools.py` only handles **one** image source.  
Your task: extend it to accept a **list** of sources and build the correct multi-image content array.

**Expected behaviour:**
- `vision_content_multi([url1, url2], "Compare...")` → list of 3 blocks: image, image, text
- `vision_content_multi([url1], "Describe...")` → list of 2 blocks: image, text (same as original)
- Works for both HTTP URLs and local file paths

In [ ]:
# Exercise 1: Modify vision_content to accept multiple image sources
# TODO: change signature to accept sources: list[str]
# TODO: loop over each source and build an image_url block
# TODO: append the text block at the end

def vision_content_multi(sources: list, text: str) -> list:
    # TODO: implement
    pass


# Uncomment to test once implemented:
# result = vision_content_multi([IMAGE_URL, SECOND_IMAGE_URL], "Compare these images")
# assert len(result) == 3, f"Expected 3 blocks, got {len(result)}"
# assert result[0]["type"] == "image_url"
# assert result[1]["type"] == "image_url"
# assert result[2]["type"] == "text"
# print("Exercise 1: PASS")

In [ ]:
# Answer Key — Exercise 1

def vision_content_multi(sources: list, text: str) -> list:
    """Build a multimodal content list from multiple image sources + a text prompt."""
    content = []
    for source in sources:
        if source.startswith("http"):
            resp = httpx.get(source, timeout=10)
            resp.raise_for_status()
            img_mime = resp.headers.get("content-type", "image/png").split(";")[0]
            img_b64 = base64.b64encode(resp.content).decode()
        else:
            from pathlib import Path
            path = Path(source)
            ext = path.suffix.lstrip(".").lower()
            img_mime = MIME_MAP.get(ext, "image/jpeg")
            img_b64 = base64.b64encode(path.read_bytes()).decode()
        content.append(
            {"type": "image_url", "image_url": {"url": f"data:{img_mime};base64,{img_b64}"}}
        )
    content.append({"type": "text", "text": text})
    return content


# Verify structure
result = vision_content_multi([IMAGE_URL, SECOND_IMAGE_URL], "What differs between these two images?")
print(f"Content blocks : {len(result)}")
print(f"Types          : {[b['type'] for b in result]}")
assert len(result) == 3
assert result[2]["type"] == "text"
print("\nKey insight: the model sees BOTH images in one turn.")
print("This is more efficient than two separate calls.")

---
### Exercise 2: Add `detail` level control

OpenAI allows a `"detail"` key on the `image_url` object to control token cost:
- `"detail": "low"` → 1 tile, ~85 tokens (fastest, cheapest)
- `"detail": "high"` → full tiling, 170 tokens per 512 px crop
- `"detail": "auto"` → model chooses based on image size (default)

Your task: extend `vision_content_multi` to accept an optional `detail` parameter and include it in each `image_url` block.

In [ ]:
# Exercise 2: Add detail level control
# TODO: add `detail: str = "auto"` parameter to vision_content_multi
# TODO: include `"detail": detail` inside each image_url dict

def vision_content_multi_v2(sources: list, text: str, detail: str = "auto") -> list:
    # TODO: implement — copy your answer from Exercise 1 and add detail
    pass


# Uncomment to test:
# low_result = vision_content_multi_v2([IMAGE_URL], "What is this?", detail="low")
# assert low_result[0]["image_url"]["detail"] == "low", "detail key missing"
# print("Exercise 2: PASS")

In [ ]:
# Answer Key — Exercise 2

def vision_content_multi_v2(sources: list, text: str, detail: str = "auto") -> list:
    """Multi-image content builder with configurable detail level."""
    content = []
    for source in sources:
        if source.startswith("http"):
            resp = httpx.get(source, timeout=10)
            resp.raise_for_status()
            img_mime = resp.headers.get("content-type", "image/png").split(";")[0]
            img_b64 = base64.b64encode(resp.content).decode()
        else:
            from pathlib import Path
            path = Path(source)
            ext = path.suffix.lstrip(".").lower()
            img_mime = MIME_MAP.get(ext, "image/jpeg")
            img_b64 = base64.b64encode(path.read_bytes()).decode()
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:{img_mime};base64,{img_b64}",
                "detail": detail,
            },
        })
    content.append({"type": "text", "text": text})
    return content


# Verify
low_result = vision_content_multi_v2([IMAGE_URL], "What is this?", detail="low")
high_result = vision_content_multi_v2([IMAGE_URL], "Describe in detail.", detail="high")

print(f"low  detail block : {low_result[0]['image_url']['detail']}")
print(f"high detail block : {high_result[0]['image_url']['detail']}")
assert low_result[0]["image_url"]["detail"] == "low"
assert high_result[0]["image_url"]["detail"] == "high"
print("\nWith detail='low': ~85 tokens per image — great for quick classification tasks.")
print("With detail='high': full tiling — necessary for reading small text in documents.")

---
### Exercise 3: Write a Dashboard Screenshot Analyser Prompt

Different image types benefit from purpose-built system prompts.  
Write a **system prompt** that instructs the model to extract three structured fields
from a **dashboard screenshot**:

1. `title` — the dashboard or chart title
2. `main_metric` — the primary KPI shown prominently
3. `date_range` — the time period the data covers

Return the answer as valid JSON.

In [ ]:
# Exercise 3: Write a system prompt for dashboard screenshot analysis.
# The model should return {"title": ..., "main_metric": ..., "date_range": ...}

# TODO: fill in the prompt string
DASHBOARD_SYSTEM_PROMPT = ""

# Uncomment to inspect once filled in:
# print(DASHBOARD_SYSTEM_PROMPT)

In [ ]:
# Answer Key — Exercise 3

DASHBOARD_SYSTEM_PROMPT = """You are a dashboard analysis assistant.
When given a screenshot of a data dashboard or chart, extract these fields:
- title: the main heading of the dashboard
- main_metric: the primary KPI or number most prominently displayed
- date_range: the time period covered (e.g. \"Jan 2024 - Mar 2024\")

Respond ONLY with valid JSON in exactly this shape:
{\"title\": \"...\", \"main_metric\": \"...\", \"date_range\": \"...\"}

Use null for any field not visible in the screenshot."""

import json as _json

# Show how this slots into a messages array
messages_preview = [
    {"role": "system", "content": DASHBOARD_SYSTEM_PROMPT.split("\n")[0] + " ..."},
    {"role": "user",   "content": "[image_url block + text block]"},
]
print("System prompt (first line):")
print(f"  {DASHBOARD_SYSTEM_PROMPT.split(chr(10))[0]}")
print("\nMessages array shape:")
print(_json.dumps(messages_preview, indent=2))
print("\nKey insight: the system prompt sets the output contract;")
print("the user turn provides the image + a minimal instruction.")

---
## Part B — Live API Calls

> **Requires `OPENAI_API_KEY`** in your `.env` file (or environment variable).

```bash
# .env
OPENAI_API_KEY=sk-...
```

**Request budget for the demo run:** three `gpt-5.4-nano` vision requests against one small public image. Check the account pricing page for current model billing.

**What this section demonstrates:**
- Fail-fast API key + connectivity check before spending tokens
- The `VisionState` LangGraph workflow compiled inline
- Running 3 different questions against the same image
- Token cost estimation

In [ ]:
# Part B · Setup — load .env, verify API key, probe connectivity
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY not set.\n"
        "Add it to your .env file or export it as an environment variable, "
        "then re-run this cell."
    )

client = OpenAI(api_key=api_key)

# Quick connectivity probe — list models is cheap and read-only
try:
    models = client.models.list()
    requested_model = "gpt-5.4-nano"
    available_models = {m.id for m in models}
    if requested_model not in available_models:
        raise RuntimeError(f"Requested model {requested_model!r} is not available to this account.")
    print("Connected to OpenAI API.")
    print(f"Requested model is available: {requested_model}")
except Exception as exc:
    raise ConnectionError(f"Cannot reach OpenAI API: {exc}") from exc

print("\nReady — Part B cells can now run.")

### The LangGraph Vision Workflow

The `src/workflow.py` file defines a minimal single-node graph using `VisionState`:

```python
class VisionState(TypedDict):
    image_source: str   # URL or local path
    question: str       # the text prompt
    answer: str         # filled by answer_node
```

State flow:

```
Input State            answer_node              Output State
────────────           ───────────────          ────────────
image_source ────────► load_image_b64()  ────► answer
question     ────────► build content     ────► (unchanged)
answer: ""              API call
```

The graph has exactly one node (`answer`) wired straight to `END`.
Client and model are passed through LangGraph's `config["configurable"]` so the node stays stateless and testable.

In [ ]:
# Build the LangGraph workflow inline — mirrors src/workflow.py exactly
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableConfig
from typing import TypedDict


class VisionState(TypedDict):
    image_source: str
    question: str
    answer: str


def _load_b64(source: str):
    """Inline helper: return (b64, mime) from URL or file path."""
    if source.startswith("http"):
        r = httpx.get(source, timeout=10)
        r.raise_for_status()
        img_mime = r.headers.get("content-type", "image/jpeg").split(";")[0]
        return base64.b64encode(r.content).decode(), img_mime
    from pathlib import Path
    path = Path(source)
    ext = path.suffix.lstrip(".").lower()
    img_mime = MIME_MAP.get(ext, "image/jpeg")
    return base64.b64encode(path.read_bytes()).decode(), img_mime


def answer_node(state: VisionState, config: RunnableConfig) -> VisionState:
    c = config["configurable"]["client"]
    model = config["configurable"].get("model", "gpt-5.4-nano")
    img_b64, img_mime = _load_b64(state["image_source"])
    content = [
        {"type": "image_url", "image_url": {"url": f"data:{img_mime};base64,{img_b64}"}},
        {"type": "text", "text": state["question"]},
    ]
    resp = c.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": content}],
        max_completion_tokens=512,
    )
    return {**state, "answer": resp.choices[0].message.content}


g = StateGraph(VisionState)
g.add_node("answer", answer_node)
g.set_entry_point("answer")
g.add_edge("answer", END)
workflow = g.compile()

print("Workflow compiled successfully.")
print(f"Nodes : {list(workflow.get_graph().nodes.keys())}")

In [ ]:
# Run 3 questions against one public image through the compiled workflow
IMAGE = (
    "https://raw.githubusercontent.com/github/explore/main/topics/"
    "python/python.png"
)

questions = [
    "What colors appear in this image?",
    "Describe the two main shapes in the image.",
    "What programming language does this image represent?",
]

cfg = {"configurable": {"client": client, "model": "gpt-5.4-nano"}}

print(f"Image: {IMAGE}\n")
print("=" * 64)
for i, q in enumerate(questions, 1):
    result = workflow.invoke(
        {"image_source": IMAGE, "question": q, "answer": ""},
        config=cfg,
    )
    print(f"\nQ{i}: {q}")
    print(f"A{i}: {result['answer']}")
    print("-" * 64)

In [ ]:
# Pretty-print Q&A pairs in a table  (requires results from the cell above)
# Re-run run-3-questions if `results` is not defined.

try:
    _results = [
        (q, workflow.invoke(
            {"image_source": IMAGE, "question": q, "answer": ""},
            config=cfg,
        )["answer"])
        for q in questions
    ]
except NameError:
    raise RuntimeError("Run the 'Build the LangGraph workflow' and setup cells first.")


def _wrap(text: str, width: int) -> list:
    words, lines, cur = text.split(), [], ""
    for w in words:
        if len(cur) + len(w) + 1 <= width:
            cur = (cur + " " + w).lstrip()
        else:
            lines.append(cur); cur = w
    if cur: lines.append(cur)
    return lines or [""]


COL = 38
print(f"{'Question':<{COL}}  {'Answer':<{COL}}")
print("-" * (COL * 2 + 2))
for q, a in _results:
    ql, al = _wrap(q, COL), _wrap(a, COL)
    for i in range(max(len(ql), len(al))):
        print(f"{ql[i] if i < len(ql) else '':<{COL}}  {al[i] if i < len(al) else '':<{COL}}")
    print()

In [ ]:
# Sandbox — try your own image and question
# Replace MY_IMAGE with any public URL or local file path.

MY_IMAGE    = IMAGE       # swap in your own URL
MY_QUESTION = "Is there any text visible? If so, what does it say?"

result = workflow.invoke(
    {"image_source": MY_IMAGE, "question": MY_QUESTION, "answer": ""},
    config=cfg,
)
print(f"Q: {MY_QUESTION}")
print(f"A: {result['answer']}")

In [ ]:
# Request footprint for the 3-question run above.
# Use your account's current model pricing for a monetary estimate.

# Image: ~240x240 px PNG with auto detail → 1 tile → ~255 tokens each call
image_tokens_per_call = 255
# Typical question: ~10 tokens
text_tokens_per_call  = 10
# Typical answer: ~80 tokens
output_tokens_per_call = 80

n_calls = len(questions)

total_input  = (image_tokens_per_call + text_tokens_per_call) * n_calls
total_output = output_tokens_per_call * n_calls

print(f"Calls                  : {n_calls}")
print(f"Input tokens / call    : {image_tokens_per_call} (image) + {text_tokens_per_call} (text)")
print(f"Total input tokens     : {total_input:,}")
print(f"Total output tokens    : {total_output:,}")
print("\nMonetary cost: consult the current pricing page for gpt-5.4-nano.")

---
## What's Next

- **OpenAI Vision docs:** https://platform.openai.com/docs/guides/vision — authoritative reference for `detail`, `max_tokens`, and multi-image limits
- **Example 139 (doc-vision-agent):** extracting structured JSON data from scanned PDF pages using vision — chains `pdf2image` → vision → Pydantic parsing
- **Extending to video frames:** sample a video with `cv2` or `ffmpeg` → extract N frames → run this workflow per frame → aggregate answers into a timeline
- **Multi-modal RAG (Example 42):** embed image *descriptions* alongside text chunks so a retriever can surface visually relevant content without storing raw images in the vector DB
- **Vision-language benchmarks:** MMMU and MMBench measure multimodal performance across disciplines; use them alongside your own representative images when evaluating a model.